# Data Viewer

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import (
    ShortTimeFFT,
    find_peaks,
    get_window,
    hilbert,
)
from tritonoa.data.reader import read_hdf5
from tritonoa.data.time import TIME_PRECISION

FIGWIDTH = 14

In [ ]:
# sensor = "3dvha"
sensor = "vla1"
# sensor = "vla2"
time_start = np.datetime64("2023-12-01 21:06:15", TIME_PRECISION)
time_end = np.datetime64("2023-12-01 21:08:15", TIME_PRECISION)

## Templates

## Filtering Results

In [ ]:
ds = read_hdf5(Path("data/acoustic/denoised") / f"{sensor}.h5").trim(time_start, time_end).filter("bandpass", [15.0, 50.0])
y_orig = ds.data[0]
y_filt = ds.data[1]
y_temp = ds.data[2]
t = ds.time_vector
fs = ds.stats.sampling_rate

### Time Series

In [ ]:
fig, axes = plt.subplots(nrows=3, figsize=(FIGWIDTH, 9), sharex=True)
ax = axes[0]
ax.plot(t, y_orig, label="Original Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Original Signal")

ax = axes[1]
ax.plot(t, y_temp, "tab:red", label="Temp Signal")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Template Signal")

ax = axes[2]
ax.plot(t, y_filt, "tab:green", label="Filtered Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Filtered Signal")

plt.tight_layout()
plt.show()

### Spectrograms

In [ ]:
window = "hann"
nperseg = 16384
hop = 8192
flim = [15, 50]


STFT = ShortTimeFFT(fs=fs, hop=hop, mfft=nperseg, win=get_window(window, nperseg))
Zxx_orig = STFT.stft(y_orig)
Zxx_orig_db = 20 * np.log10(np.abs(Zxx_orig))
tvec = STFT.t(len(y_orig))
f = STFT.f

vmax = np.max(Zxx_orig_db)
vmin = vmax - 60

plt.figure(figsize=(FIGWIDTH, 12))
plt.subplot(311)
plt.pcolormesh(tvec, f, Zxx_orig_db, shading="gouraud", vmin=vmin, vmax=vmax)
plt.ylim(flim)
plt.colorbar(label="Magnitude (dB)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("STFT of Original Signal")


STFT = ShortTimeFFT(fs=fs, hop=hop, mfft=nperseg, win=get_window(window, nperseg))
Zxx_filt = STFT.stft(y_filt)
Zxx_filt_db = 20 * np.log10(np.abs(Zxx_filt))
tvec = STFT.t(len(y_filt))
f = STFT.f

plt.subplot(312)
plt.pcolormesh(tvec, f, Zxx_filt_db, shading="gouraud", vmin=vmin, vmax=vmax)
plt.ylim(flim)
plt.colorbar(label="Magnitude (dB)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("STFT of Filtered Signal")

Zxx_diff = Zxx_filt_db - Zxx_orig_db

plt.subplot(313)
plt.pcolormesh(tvec, f, Zxx_diff, shading="gouraud", cmap="bwr", vmin=-30, vmax=30)
plt.ylim(flim)
plt.colorbar(label="Magnitude Difference (dB)")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Difference")

plt.tight_layout()
plt.show()

## Pulse Compression Results

In [ ]:
# sensor = "3dvha"
sensor = "vla1"
# sensor = "vla2"
time_starts = np.array(
    [
        np.datetime64("2023-12-01 21:06:00", TIME_PRECISION),
        np.datetime64("2023-12-01 21:26:00", TIME_PRECISION),
        np.datetime64("2023-12-01 21:46:00", TIME_PRECISION),
        np.datetime64("2023-12-01 22:06:00", TIME_PRECISION),
    ]
)
time_ends = time_starts + np.timedelta64(20, "m")

In [ ]:
segment = 3
ds = (
    read_hdf5(Path("data/acoustic/denoised") / f"{sensor}_pc.h5")
    .trim(time_starts[segment], time_ends[segment])
    # .filter("bandpass", [15.0, 30.0])
)
y_orig = ds.data[0]
y_filt = ds.data[1]
pc_orig_type1 = ds.data[2]
pc_orig_type2 = ds.data[3]
pc_dn_type1 = ds.data[4]
pc_dn_type2 = ds.data[5]
t = ds.time_vector
fs = ds.stats.sampling_rate

In [ ]:
fig, axes = plt.subplots(nrows=6, figsize=(FIGWIDTH, 15), sharex=True)

ax = axes[0]
ax.plot(t, y_orig, label="Original Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Original Signal")

ax = axes[1]
ax.plot(t, y_filt, label="Filtered Signal")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Filtered Signal")

ax = axes[2]
ax.plot(t, pc_orig_type1, "tab:red", label="PC Original Type 1")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Original Type 1")

ax = axes[3]
ax.plot(t, pc_orig_type2, "tab:green", label="PC Original Type 2")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Original Type 2")

ax = axes[4]
ax.plot(t, pc_dn_type1, "tab:orange", label="PC Denoised Type 1")
ax.grid()
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Denoised Type 1")

ax = axes[5]
ax.plot(t, pc_dn_type2, "tab:purple", label="PC Denoised Type 2")
ax.grid()
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title("Pulse Compression Denoised Type 2")

plt.tight_layout()
plt.show()

## Peak Finding

In [ ]:
start, end = time_starts[0], time_ends[-1]

ds = (
    read_hdf5(Path("data/acoustic/denoised") / f"{sensor}_pc.h5")
    .trim(start, end)
    .filter("bandpass", [15.0, 30.0])
)
pc_dn_type2 = ds.data[5]
t = ds.time_vector
fs = ds.stats.sampling_rate
del ds

threshold = 0.3
distance = fs * 7
cf = np.abs(hilbert(pc_dn_type2))
cf /= np.max(cf)
peaks = find_peaks(cf, height=threshold, distance=distance)[0]

plt.figure(figsize=(FIGWIDTH, 6))
plt.plot(t, cf, label="Envelope of PC Denoised Type 2")
plt.axhline(threshold, color="red", linestyle="--", label="Peak Threshold")
plt.plot(t[peaks], cf[peaks], "x", label="Peaks")
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Envelope of Pulse Compression Denoised Type 2")

plt.tight_layout()
plt.show()